# Library Import

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import joblib

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.max_columns", None)

# Load Feature Selectioned Dataset

In [ ]:
# =========================================================
# LOAD FEATURE SELECTION METADATA
# =========================================================

import json

with open( "/content/selected_features.json", "r") as f:
    feature_metadata = json.load(f)

SELECTED_FEATURES = (feature_metadata["selected_features"])
TARGET = (feature_metadata["target"])

print("=" * 60)
print("FEATURE METADATA LOADED")
print("=" * 60)
print("Selected Features:")
print(SELECTED_FEATURES)
print("\nTarget:")
print(TARGET)

FEATURE METADATA LOADED
Selected Features:
['days_for_shipment_(scheduled)', 'benefit_per_order', 'category_name', 'customer_id', 'customer_segment', 'market', 'order_date_(dateorders)', 'order_id', 'order_item_quantity', 'sales', 'order_region', 'shipping_mode']

Target:
late_delivery_risk


In [ ]:
# =========================================================
# 3. LOAD RAW DATASET
# =========================================================

df = pd.read_csv(
    "/content/DataCoSupplyChainDataset.csv",
    encoding="latin1"
)

print("=" * 60)
print("RAW DATASET LOADED")
print("=" * 60)

print(df.shape)

display(df.head())

RAW DATASET LOADED
(180519, 53)


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,Customer Country,Customer Email,Customer Fname,Customer Id,Customer Lname,Customer Password,Customer Segment,Customer State,Customer Street,Customer Zipcode,Department Id,Department Name,Latitude,Longitude,Market,Order City,Order Country,Order Customer Id,order date (DateOrders),Order Id,Order Item Cardprod Id,Order Item Discount,Order Item Discount Rate,Order Item Id,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Cally,20755,Holloway,XXXXXXXXX,Consumer,PR,5365 Noble Nectar Island,725.0,2,Fitness,18.251453,-66.037056,Pacific Asia,Bekasi,Indonesia,20755,1/31/2018 22:56,77202,1360,13.110000,0.04,180517,327.75,0.29,1,327.75,314.640015,91.250000,Southeast Asia,Java Occidental,COMPLETE,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Irene,19492,Luna,XXXXXXXXX,Consumer,PR,2679 Rustic Loop,725.0,2,Fitness,18.279451,-66.037064,Pacific Asia,Bikaner,India,19492,1/13/2018 12:27,75939,1360,16.389999,0.05,179254,327.75,-0.80,1,327.75,311.359985,-249.089996,South Asia,Rajastán,PENDING,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,EE. UU.,XXXXXXXXX,Gillian,19491,Maldonado,XXXXXXXXX,Consumer,CA,8510 Round Bear Gate,95125.0,2,Fitness,37.292233,-121.881279,Pacific Asia,Bikaner,India,19491,1/13/2018 12:06,75938,1360,18.030001,0.06,179253,327.75,-0.80,1,327.75,309.720001,-247.779999,South Asia,Rajastán,CLOSED,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,EE. UU.,XXXXXXXXX,Tana,19490,Tate,XXXXXXXXX,Home Office,CA,3200 Amber Bend,90027.0,2,Fitness,34.125946,-118.291016,Pacific Asia,Townsville,Australia,19490,1/13/2018 11:45,75937,1360,22.940001,0.07,179252,327.75,0.08,1,327.75,304.809998,22.860001,Oceania,Queensland,COMPLETE,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Orli,19489,Hendricks,XXXXXXXXX,Corporate,PR,8671 Iron Anchor Corners,725.0,2,Fitness,18.253769,-66.037048,Pacific Asia,Townsville,Australia,19489,1/13/2018 11:24,75936,1360,29.500000,0.09,179251,327.75,0.45,1,327.75,298.250000,134.210007,Oceania,Queensland,PENDING_PAYMENT,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


# Split Data

In [ ]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

In [ ]:
# =========================================================
# 5. DEFINE X AND y
# =========================================================

X = df[SELECTED_FEATURES].copy()
y = df[TARGET].copy()

print("X Shape:", X.shape)
print("y Shape:", y.shape)

X Shape: (180519, 12)
y Shape: (180519,)


In [ ]:
# =========================================================
# 6. TRAIN TEST SPLIT
# =========================================================

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42,
)

split = gss.split(
    X, y,
    groups=X["customer_id"]
)

train_idx, test_idx = next(split)
X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

print("=" * 60)
print("TRAIN TEST SPLIT")
print("=" * 60)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)


train_ratio = y_train.mean()
test_ratio = y_test.mean()

print(f"Train Target Ratio : {train_ratio:.4f}")
print(f"Test Target Ratio  : {test_ratio:.4f}")
print(f"Absolute Difference  : {abs(train_ratio - test_ratio):.4f}")

TRAIN TEST SPLIT
X_train: (143757, 12)
X_test : (36762, 12)
Train Target Ratio : 0.5498
Test Target Ratio  : 0.5423
Absolute Difference  : 0.0076


Cek unseen customer

In [ ]:
# =========================================================
# CUSTOMER OVERLAP VALIDATION
# =========================================================

train_customers = set(
    X_train["customer_id"]
)

test_customers = set(
    X_test["customer_id"]
)

overlap = (
    train_customers
    .intersection(test_customers)
)

print("=" * 60)
print("CUSTOMER OVERLAP VALIDATION")
print("=" * 60)

print(
    f"Overlap Customer Count: {len(overlap)}"
)

CUSTOMER OVERLAP VALIDATION
Overlap Customer Count: 0


# Feature Engineering

In [ ]:
# 6. Decreasing potentiant leakge model
train_df = pd.concat(
    [X_train, y_train],
    axis=1
)

print("=" * 60)
print("SAFE FEATURE ENGINEERING")
print("=" * 60)

SAFE FEATURE ENGINEERING


Temporal Features

In [ ]:
# Convert 'order_date_(dateorders)' to datetime type
X_train['order_date_(dateorders)'] = pd.to_datetime(X_train['order_date_(dateorders)'])
X_test['order_date_(dateorders)'] = pd.to_datetime(X_test['order_date_(dateorders)'])

# order month
X_train["order_month"] = (
    X_train["order_date_(dateorders)"].dt.month
)

X_test["order_month"] = (
    X_test["order_date_(dateorders)"].dt.month
)

# order day of week
X_train["order_dayofweek"] = (
    X_train["order_date_(dateorders)"].dt.dayofweek
)

X_test["order_dayofweek"] = (
    X_test["order_date_(dateorders)"].dt.dayofweek
)

# weekend order
X_train["is_weekend_order"] = (
    X_train["order_dayofweek"]
    .isin([5, 6]).astype(int)
)

X_test["is_weekend_order"] = (
    X_test["order_dayofweek"]
    .isin([5, 6]).astype(int)
)

Shipping Mode Risk

In [ ]:
shipping_mode_risk = (
    train_df
    .groupby("shipping_mode")
    [TARGET]
    .mean()
)

X_train["shipping_mode_risk"] = (
    X_train["shipping_mode"]
    .map(shipping_mode_risk)
)

X_test["shipping_mode_risk"] = (
    X_test["shipping_mode"]
    .map(shipping_mode_risk)
)

global_shipping_risk = (
    y_train.mean()
)

X_test["shipping_mode_risk"] = (
    X_test["shipping_mode_risk"]
    .fillna(global_shipping_risk)
)

Region Risk

In [ ]:
region_risk = (
    train_df.groupby("order_region")[TARGET].mean()
)

X_train["avg_delay_by_region"] = (
    X_train["order_region"].map(region_risk)
)

X_test["avg_delay_by_region"] = (
    X_test["order_region"].map(region_risk)
)

global_region_risk = (y_train.mean())

X_test["avg_delay_by_region"] = (
    X_test["avg_delay_by_region"].fillna(global_region_risk)
)

display(region_risk.head(10))

,late_delivery_risk
order_region,
Canada,0.482984
Caribbean,0.529330
Central Africa,0.589917
Central America,0.549852
Central Asia,0.614035
East Africa,0.559564
East of USA,0.551987
Eastern Asia,0.542747
Eastern Europe,0.565217


Category Risk

In [ ]:
category_risk = (
    train_df.groupby("category_name")[TARGET]
    .mean()
)

X_train["category_delay_risk"] = (
    X_train["category_name"].map(category_risk)
)

X_test["category_delay_risk"] = (
    X_test["category_name"].map(category_risk)
)

global_category_risk = (y_train.mean())

X_test["category_delay_risk"] = (
    X_test["category_delay_risk"].fillna(global_category_risk)
)

display(category_risk.head(10))

,late_delivery_risk
category_name,
Accessories,0.561317
As Seen on TV!,0.607143
Baby,0.530726
Baseball & Softball,0.543353
Basketball,0.509091
Books,0.558642
Boxing & MMA,0.560000
CDs,0.509174
Cameras,0.575492


Customer Order Count

In [ ]:
customer_order_count = (
    train_df.groupby("customer_id")["order_id"]
    .count()
)

X_train["customer_order_count"] = (
    X_train["customer_id"].map(customer_order_count)
)

X_test["customer_order_count"] = (
    X_test["customer_id"].map(customer_order_count)
)

X_test["customer_order_count"]

display(customer_order_count.head())

,order_id
customer_id,
1,1
2,10
3,18
5,7
6,15


Customer Late Rate

In [ ]:
# =========================================================
# 11. BAYESIAN CUSTOMER LATE RATE
# =========================================================

global_late_rate = y_train.mean()

customer_stats = (
    train_df.groupby("customer_id")
    [TARGET].agg(["count", "mean"])
)

customer_stats.columns = [
    "transaction_count",
    "raw_late_rate"
]

SMOOTHING_FACTOR = 10

customer_stats[
    "customer_late_rate"
] = ((customer_stats["transaction_count"] *
     customer_stats["raw_late_rate"])
    +
    (SMOOTHING_FACTOR * global_late_rate)
) / (customer_stats["transaction_count"] + SMOOTHING_FACTOR)

In [ ]:
# =========================================================
# 12. APPLY CUSTOMER LATE RATE
# =========================================================

X_train["customer_late_rate"] = (
    X_train["customer_id"]
    .map(customer_stats["customer_late_rate"])
)

X_test["customer_late_rate"] = (
    X_test["customer_id"]
    .map(customer_stats["customer_late_rate"])
)

X_test["customer_late_rate"] = (
    X_test["customer_late_rate"]
    .fillna(global_late_rate)
)

In [ ]:
# =========================================================
# 14. REMOVE HELPER COLUMN
# =========================================================

X_train = X_train.drop(columns=["customer_id"])
X_test = X_test.drop(columns=["customer_id"])

print("=" * 60)
print("HELPER COLUMN REMOVED")
print("=" * 60)

HELPER COLUMN REMOVED


Training Feature

In [ ]:
# =========================================================
# 15. FEATURE VALIDATION
# =========================================================

print("=" * 60)
print("FEATURE VALIDATION")
print("=" * 60)

print("\nMissing Values:")
print(X_train.isnull().sum())

print("\nTrain Shape:")
print(X_train.shape)

print("\nTest Shape:")
print(X_test.shape)

FEATURE VALIDATION

Missing Values:
days_for_shipment_(scheduled)    0
benefit_per_order                0
category_name                    0
customer_segment                 0
market                           0
order_date_(dateorders)          0
order_id                         0
order_item_quantity              0
sales                            0
order_region                     0
shipping_mode                    0
order_month                      0
order_dayofweek                  0
is_weekend_order                 0
shipping_mode_risk               0
avg_delay_by_region              0
category_delay_risk              0
customer_order_count             0
customer_late_rate               0
dtype: int64

Train Shape:
(143757, 19)

Test Shape:
(36762, 19)


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 53 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   type                           180519 non-null  object 
 1   days_for_shipping_(real)       180519 non-null  int64  
 2   days_for_shipment_(scheduled)  180519 non-null  int64  
 3   benefit_per_order              180519 non-null  float64
 4   sales_per_customer             180519 non-null  float64
 5   delivery_status                180519 non-null  object 
 6   late_delivery_risk             180519 non-null  int64  
 7   category_id                    180519 non-null  int64  
 8   category_name                  180519 non-null  object 
 9   customer_city                  180519 non-null  object 
 10  customer_country               180519 non-null  object 
 11  customer_email                 180519 non-null  object 
 12  customer_fname                

In [38]:
RAW_FEATURES = [
    "days_for_shipment_(scheduled)",
    "benefit_per_order",
    "category_name",
    "customer_segment",
    "market",
    "order_date_(dateorders)",
    "order_item_quantity",
    "sales",
    "order_region",
    "shipping_mode"
]

ENGINEERED_FEATURES = [

    # temporal features
    "order_month",
    "order_dayofweek",
    "is_weekend_order",

    # historical aggregation features
    "shipping_mode_risk",
    "avg_delay_by_region",
    "category_delay_risk",
    "customer_order_count",
    "customer_late_rate"
]

TRAINING_FEATURES = (
    RAW_FEATURES +
    ENGINEERED_FEATURES
)

TARGET = "late_delivery_risk"

print("RAW FEATURES:")
display(RAW_FEATURES)

print("\nENGINEERED FEATURES:")
display(ENGINEERED_FEATURES)

print("\nTOTAL TRAINING FEATURES:")
print(len(TRAINING_FEATURES))

RAW FEATURES:


['days_for_shipment_(scheduled)',
 'benefit_per_order',
 'category_name',
 'customer_segment',
 'market',
 'order_date_(dateorders)',
 'order_item_quantity',
 'sales',
 'order_region',
 'shipping_mode']


ENGINEERED FEATURES:


['order_month',
 'order_dayofweek',
 'is_weekend_order',
 'shipping_mode_risk',
 'avg_delay_by_region',
 'category_delay_risk',
 'customer_order_count',
 'customer_late_rate']


TOTAL TRAINING FEATURES:
18


Validation Feature

In [39]:
missing_training_features = [
    col for col in TRAINING_FEATURES
    if col not in X_train.columns
]

if len(missing_training_features) == 0:
    print("All training features exist.")
else:
    print("Missing Features:")
    display(missing_training_features)

All training features exist.


Feature Cardinality Check

In [ ]:
categorical_features = [
    "shipping_mode",
    "market",
    "order_region",
    "customer_segment",
    "category_name"
]

cardinality_df = pd.DataFrame({
    "feature": categorical_features,
    "unique_count": [
        df[col].nunique()
        for col in categorical_features
    ]
})

display(cardinality_df)

,feature,unique_count
0,shipping_mode,4
1,market,5
2,order_region,23
3,customer_segment,3
4,category_name,41


In [ ]:
# =========================================================
# MAPPING COVERAGE VALIDATION
# =========================================================

print("=" * 60)
print("MAPPING COVERAGE VALIDATION")
print("=" * 60)

print("\nTrain Missing:")
print(
    X_train[
        [
            "customer_order_count",
            "customer_late_rate"
        ]
    ]
    .isnull()
    .sum()
)

print("\nTest Missing:")
print(
    X_test[
        [
            "customer_order_count",
            "customer_late_rate"
        ]
    ]
    .isnull()
    .sum()
)

MAPPING COVERAGE VALIDATION

Train Missing:


KeyError: "['customer_late_rate'] not in index"

In [ ]:
# =========================================================
# FINAL TRAINING DATASET
# =========================================================

final_columns = TRAINING_FEATURES + [TARGET]

df_final = df[final_columns].copy()

print("=" * 60)
print("FINAL TRAINING DATASET")
print("=" * 60)

print(f"Final Shape: {df_final.shape}")

print("\nFinal Columns:")
display(df_final.columns.tolist())

print("\nSample Data:")
display(df_final.head())

KeyError: "['shipping_mode_risk', 'avg_delay_by_region', 'category_delay_risk', 'customer_order_count'] not in index"

# Define Features & Target

In [ ]:
# =========================================================
# 7. IDENTIFY FEATURE TYPES
# =========================================================

categorical_features = [
    "category_name",
    "customer_segment",
    "market",
    "order_region",
    "shipping_mode"
]

numerical_features = [
    "days_for_shipment_(scheduled)",
    "sales",
    "shipping_mode_risk",
    "avg_delay_by_region",
    "category_delay_risk",
    "customer_order_count",
    "customer_late_rate",
    "urgent_shipping"
]

print("Categorical Features:")
display(categorical_features)

print("\nNumerical Features:")
display(numerical_features)

In [ ]:
# =========================================================
# 3. DEFINE FEATURES & TARGET
# =========================================================

RAW_FEATURES = [
    "days_for_shipment_(scheduled)",
    "category_name",
    "customer_segment",
    "market",
    "sales",
    "order_region",
    "shipping_mode"
]

ENGINEERED_FEATURES = [
    "shipping_mode_risk",
    "avg_delay_by_region",
    "category_delay_risk",
    "customer_order_count",
    "customer_late_rate",
    "urgent_shipping"
]

TRAINING_FEATURES = (
    RAW_FEATURES +
    ENGINEERED_FEATURES
)

TARGET = "late_delivery_risk"

print("Total Features:", len(TRAINING_FEATURES))

In [ ]:
# =========================================================
# 4. VALIDATE REQUIRED FEATURES
# =========================================================

missing_features = [
    col for col in TRAINING_FEATURES + [TARGET]
    if col not in df.columns
]

if len(missing_features) == 0:
    print("All required features exist.")
else:
    print("Missing Features:")
    display(missing_features)

# Preprocessing

In [ ]:
# =========================================================
# 8. PREPROCESSING PIPELINE
# =========================================================

categorical_transformer = Pipeline(steps=[
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )
])

numerical_transformer = Pipeline(steps=[
    (
        "imputer",
        SimpleImputer(strategy="median")
    )
])

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            categorical_transformer,
            categorical_features
        ),
        (
            "num",
            numerical_transformer,
            numerical_features
        )
    ]
)

print("Preprocessing pipeline created.")

In [ ]:
# =========================================================
# 9. FIT PREPROCESSOR ON TRAIN ONLY
# =========================================================

X_train_processed = preprocessor.fit_transform(
    X_train
)

X_test_processed = preprocessor.transform(
    X_test
)

print("=" * 60)
print("PREPROCESSING COMPLETE")
print("=" * 60)

print("Processed X_train:", X_train_processed.shape)
print("Processed X_test :", X_test_processed.shape)

In [ ]:
# =========================================================
# 10. GET FEATURE NAMES
# =========================================================

encoded_feature_names = (
    preprocessor.named_transformers_["cat"]
    .named_steps["encoder"]
    .get_feature_names_out(categorical_features)
)

final_feature_names = (
    list(encoded_feature_names) +
    numerical_features
)

print("Total Final Features:")
print(len(final_feature_names))

display(final_feature_names[:20])

In [ ]:
import pandas as pd

# =========================================================
# 11. CONVERT TO DATAFRAME
# =========================================================

X_train_processed_df = pd.DataFrame(
    X_train_processed.toarray(), # Convert sparse matrix to dense array
    columns=final_feature_names,
    index=X_train.index
)

X_test_processed_df = pd.DataFrame(
    X_test_processed.toarray(), # Convert sparse matrix to dense array
    columns=final_feature_names,
    index=X_test.index
)

display(X_train_processed_df.head())

In [ ]:
# =========================================================
# 12. SAVE PREPROCESSED DATA
# =========================================================

os.makedirs("/content/data/processed", exist_ok=True)

X_train_processed_df.to_csv(
    "/content/data/processed/X_train.csv",
    index=False
)

X_test_processed_df.to_csv(
    "/content/data/processed/X_test.csv",
    index=False
)

y_train.to_csv(
    "/content/data/processed/y_train.csv",
    index=False
)

y_test.to_csv(
    "/content/data/processed/y_test.csv",
    index=False
)

print("Preprocessed datasets saved.")

NameError: name 'X_train_processed_df' is not defined

# Result

In [ ]:
# =========================================================
# SAVE FEATURE ENGINEERED DATASET
# =========================================================

import os
import joblib

# create directory
os.makedirs("/content/processed", exist_ok=True)
os.makedirs("/content/metadata", exist_ok=True)
os.makedirs("/content/artifacts/metadata", exist_ok=True)

# save dataset
dataset_path = "/content/processed/feature_engineered.csv"

df_final.to_csv(dataset_path, index=False)

print("=" * 60)
print("DATASET SAVED")
print("=" * 60)

print(f"Saved dataset to:\n{dataset_path}")

# save metadata
metadata = {
    "training_features": TRAINING_FEATURES,
    "raw_features": RAW_FEATURES,
    "engineered_features": ENGINEERED_FEATURES,
    "forbidden_features": FORBIDDEN_FEATURES,
    "target": TARGET
}

metadata_path = "/content/artifacts/metadata/feature_metadata.pkl"

joblib.dump(metadata, metadata_path)

print(f"\nSaved metadata to:\n{metadata_path}")

NameError: name 'df_final' is not defined

In [ ]:
# =========================================================
# 13. SAVE PREPROCESSOR ARTIFACTS
# =========================================================

os.makedirs("/content/artifacts/encoders", exist_ok=True)
os.makedirs("/content/artifacts/metadata", exist_ok=True)

joblib.dump(
    preprocessor,
    "/content/artifacts/encoders/preprocessor.pkl"
)

joblib.dump(
    final_feature_names,
    "/content/artifacts/metadata/feature_columns.pkl"
)

print("Preprocessor artifacts saved.")

In [ ]:
# =========================================================
# 14. PREPROCESSING SUMMARY
# =========================================================

print("""
=================================================
PREPROCESSING SUMMARY
=================================================

1. Feature engineered dataset loaded
2. Features validated
3. Train-test split completed
4. Missing values handled
5. Categorical encoding completed
6. Numerical preprocessing completed
7. Feature names preserved
8. Preprocessed datasets saved
9. Preprocessor artifacts saved

Next Step:
03_modeling.ipynb
=================================================
""")

# Summary

| Feature                                                   | Status                 | Alasan Dihapus / Direvisi                                                                                                                                                                                                                  |
| --------------------------------------------------------- | ---------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| `high_value_order`                                        | ❌ Remove               | Threshold quantile bersifat arbitrer, separation risk sangat kecil, dan redundant dengan `Sales` karena tree model dapat membuat split otomatis pada numeric feature.                                                                      |
| `urgent_shipping`                                         | ❌ Remove               | EDA menunjukkan `0`, `1`, `2`, dan `4` hari memiliki perilaku berbeda. Binary flag (`<=2`) justru menghilangkan semantic operational tier. Diganti dengan penggunaan langsung `days_for_shipment_(scheduled)` sebagai categorical feature. |
|
| `raw customer_late_rate`                                  | ❌ Remove Raw Version   | Dibuat sebelum split sehingga leakage, serta sangat tidak stabil untuk customer dengan histori kecil. Diganti dengan Bayesian-smoothed train-only aggregation.                                                                             |
| `distance_proxy`                                          | ❌ Remove               | Perhitungan sebelumnya bukan actual geographic distance (`sqrt(lat² + lon²)`), sehingga tidak valid secara geospatial.                                                                                                                     |
| `Sales` thresholding (`high_value_order`)                 | ❌ Remove Thresholding  | EDA menunjukkan tidak ada nonlinear separation signifikan antar quantile sales terhadap risiko keterlambatan. Raw `Sales` lebih masuk akal jika tetap ingin mempertahankan financial context.                                              |
| `days_for_shipment_(scheduled)` sebagai numerical urgency | 🔄 Revised             | Awalnya dianggap numeric urgency feature, tetapi setelah EDA terbukti lebih tepat diperlakukan sebagai categorical operational service tier (`same-day`, `1-day`, `2-day`, `standard`).                                                    |
| `customer_late_rate`                                      | 🔄 Revised             | Tetap dipakai tetapi harus melalui Bayesian smoothing dan train-only aggregation agar leakage dan instability berkurang.                                                                                                                   |
